# FoldPipe 1QLX Prion Case Study

This educational case study parses the bundled human prion protein structure (PDB 1QLX), creates several local tensor shards, streams them through FoldPipe, and trains a tiny coordinate denoiser.

**Scientific boundary:** the generated frames are small Gaussian perturbations around one experimental structure. They are not a molecular-dynamics trajectory, and the toy network is not a validated molecular force field. The notebook demonstrates the data path only.

## 1. Install FoldPipe

Use the released GitHub tag. From a cloned repository, `%pip install -e ..` is also suitable.

In [ ]:
%pip install -q "foldpipe @ git+https://github.com/aviatorlf/FoldPipe.git@v0.3.0"

## 2. Parse the bundled 1QLX structure

Only standard PDB `ATOM` records and their Cartesian coordinates are needed for this tutorial.

In [ ]:
from pathlib import Path
import tempfile
import torch

ELEMENT_TO_Z = {"H": 1, "C": 6, "N": 7, "O": 8, "P": 15, "S": 16}

def find_1qlx():
    candidates = [
        Path("data/prion/raw/1QLX.pdb"),
        Path("../data/prion/raw/1QLX.pdb"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Run this notebook from the FoldPipe repository or its notebooks directory.")

def parse_atoms(pdb_path):
    atomic_numbers, coordinates = [], []
    for line in pdb_path.read_text(encoding="utf-8").splitlines():
        if not line.startswith("ATOM") or line[16:17] not in {" ", "A"}:
            continue
        element = line[76:78].strip().upper()
        if not element:
            element = "".join(ch for ch in line[12:16] if ch.isalpha())[:1].upper()
        if element not in ELEMENT_TO_Z:
            continue
        atomic_numbers.append(ELEMENT_TO_Z[element])
        coordinates.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return torch.tensor(atomic_numbers), torch.tensor(coordinates, dtype=torch.float32)

pdb_path = find_1qlx()
atomic_numbers, reference_positions = parse_atoms(pdb_path)
print(f"loaded {len(atomic_numbers)} atoms from {pdb_path.name}")
print("coordinate tensor:", tuple(reference_positions.shape))

## 3. Create temporary tutorial shards

The fixed seed makes the perturbations reproducible. Each `.pt` file is one shard of independent noisy copies of the same structure.

In [ ]:
shard_dir = Path(tempfile.mkdtemp(prefix="foldpipe-1qlx-"))
generator = torch.Generator().manual_seed(17)
frames_per_shard = 128
num_shards = 4

for shard_index in range(num_shards):
    noise = 0.05 * torch.randn(
        frames_per_shard, *reference_positions.shape, generator=generator
    )
    frames = reference_positions.unsqueeze(0) + noise
    torch.save(frames, shard_dir / f"checkpoint_batch_{shard_index:04d}.pt")

print(f"wrote {num_shards} temporary shards to {shard_dir}")

## 4. Stream the shards through FoldPipe

This tutorial-only local source implements the same two methods as the included Hugging Face and Google Drive sources. The loader logic does not change when the backend changes.

In [ ]:
from foldpipe import AsyncFoldPipeLoader
from foldpipe.sources import Source

class LocalShardSource(Source):
    def __init__(self, root):
        self.root = Path(root)

    def iter_files(self):
        yield from sorted(self.root.glob("checkpoint_batch_*.pt"))

    def download_chunk(self, identifier):
        try:
            return torch.load(identifier, map_location="cpu", weights_only=True)
        except TypeError:  # Compatibility with older supported PyTorch releases.
            return torch.load(identifier, map_location="cpu")

source = LocalShardSource(shard_dir)
loader = AsyncFoldPipeLoader(source=source, batch_size=16)
first_batch = next(iter(loader))
print("first streamed batch:", tuple(first_batch.shape))

## 5. Train a small coordinate denoiser

The network learns to map a perturbed coordinate to the corresponding reference coordinate. This is intentionally a lightweight software demonstration, not a physical potential.

In [ ]:
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
correction_model = nn.Sequential(
    nn.Linear(3, 32),
    nn.SiLU(),
    nn.Linear(32, 3),
).to(device)
optimizer = torch.optim.Adam(correction_model.parameters(), lr=3e-3)
criterion = nn.MSELoss()

loader = AsyncFoldPipeLoader(source=LocalShardSource(shard_dir), batch_size=16)
losses, frames_seen = [], 0
for noisy_positions in loader:
    noisy_positions = noisy_positions.to(device)
    target = reference_positions.to(device).unsqueeze(0).expand_as(noisy_positions)
    optimizer.zero_grad()
    denoised_positions = noisy_positions + correction_model(noisy_positions)
    loss = criterion(denoised_positions, target)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    frames_seen += len(noisy_positions)

print(f"device: {device}; frames streamed: {frames_seen}; final batch loss: {losses[-1]:.6f}")

## 6. Clean up and move to real shards

The temporary data can be deleted after the demonstration. For a real study, generate trajectories with a validated simulation protocol, attach provenance and license metadata, save bounded shards, and replace `LocalShardSource` with `HuggingFaceSource` or `GoogleDriveSource`.

In [ ]:
import shutil
shutil.rmtree(shard_dir)
print("removed temporary tutorial shards")